# Kronos Test Command Center

Interactive research cockpit for strategy backtesting and visualization.

### 0. Full Auto-Bootstrap (Colab Only)
Run this cell if you are using Google Colab to automatically setup the repository, requirements, and model.

In [ ]:
import sys
import os
from pathlib import Path
from getpass import getpass

def bootstrap():
    if 'google.colab' in sys.modules:
        print("🚀 Detected Google Colab. Starting automatic environment setup...")
        
        # 1. Handle GitHub Token (Securely)
        token = None
        try:
            from google.colab import userdata
            token = userdata.get('GITHUB_TOKEN')
        except:
            pass

        if not token:
            print("GitHub Token not found in Secret 'GITHUB_TOKEN'.")
            token = getpass("Please enter your GitHub Token: ")

        # 2. Repo Definitions
        if token:
            repo_kronos = f"https://{token}@github.com/shiyu-coder/Kronos.git"
            repo_test = f"https://{token}@github.com/PavelPetrovich87/kronos_test.git"
        else:
            repo_kronos = "https://github.com/shiyu-coder/Kronos.git"
            repo_test = "https://github.com/PavelPetrovich87/kronos_test.git"
        
        # 3. Clone Repositories
        if not Path('/content/kronos_test').exists():
            print("Cloning kronos_test repository...")
            !git clone {repo_test} /content/kronos_test
        
        if not Path('/content/Kronos').exists():
            print("Cloning Kronos model repository...")
            !git clone {repo_kronos} /content/Kronos
            
        # 4. Add paths to sys.path
        project_path = '/content/kronos_test'
        if project_path not in sys.path: sys.path.append(project_path)
        
        # 5. Install Dependencies & Resolve Binary Incompatibilities
        # We force upgrade numpy and pandas to ensure C-header compatibility in Python 3.12
        print("Installing and upgrading dependencies...")
        !pip install --upgrade -q numpy pandas
        !pip install -q -r /content/kronos_test/requirements.txt --ignore-installed numpy
        !pip install -q plotly ipywidgets yfinance --upgrade
        
        # 6. Fix for Python 3.12 'imp' removal
        if sys.version_info >= (3, 12):
            from types import ModuleType
            import importlib
            imp_mock = ModuleType("imp")
            imp_mock.reload = importlib.reload
            sys.modules["imp"] = imp_mock
            print("Applied Python 3.12 compatibility patch for autoreload.")

        print("✅ Setup Complete! If you see 'ValueError: numpy.dtype size changed', please restart the runtime and run all cells again.")
        
        # Auto-CD into the project folder for path resolution
        %cd /content/kronos_test

bootstrap()

### 1. Setup & Imports

In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import ipywidgets as widgets
from datetime import datetime, timedelta
from IPython.display import display, clear_output

from lib.data_loader import DataLoader
from lib.strategy_factory import StrategyFactory
from lib.backtester import Backtester
from lib.visualizer import Visualizer, TestSession
from lib.reporting import ExperimentLogger, PerformanceMetrics

### 2. Parametric Cockpit
Configure your backtest parameters below.

In [ ]:
# Widgets Definition
symbol_dropdown = widgets.Dropdown(
    options=['BTC-USD', 'ETH-USD'],
    value='BTC-USD',
    description='Symbol:',
    style={'description_width': 'initial'}
)

mode_toggle = widgets.ToggleButtons(
    options=['mock', 'real'],
    value='mock',
    description='Inference Mode:',
    tooltips=['Fast logic test', 'AI Model inference'],
)

threshold_slider = widgets.FloatSlider(
    value=0.01,
    min=0.001,
    max=0.05,
    step=0.001,
    description='Signal Threshold:',
    readout_format='.3f',
    style={'description_width': 'initial'}
)

capital_input = widgets.FloatText(
    value=10000.0,
    description='Initial Capital:',
    style={'description_width': 'initial'}
)

run_button = widgets.Button(
    description='Run Backtest',
    button_style='success',
    icon='play'
)

output_area = widgets.Output()

# Layout
cockpit = widgets.VBox([
    widgets.HBox([symbol_dropdown, mode_toggle]),
    widgets.HBox([threshold_slider, capital_input]),
    run_button
])

display(cockpit, output_area)

def on_run_clicked(b):
    with output_area:
        clear_output(wait=True)
        print(f"🚀 Initializing backtest for {symbol_dropdown.value}...")
        
        # 1. Load Data (Small Data fallback for CPU safety)
        loader = DataLoader()
        df = loader.load_ohlcv(symbol_dropdown.value)
        df = df.tail(500)  # Constraint: Small Data for speed
        
        # 2. Setup Strategy
        factory = StrategyFactory()
        strategy = factory.create_strategy(mode=mode_toggle.value)
        
        # 3. Predict & Backtest
        signals = strategy.generate_signals(df)
        backtester = Backtester(initial_capital=capital_input.value)
        result = backtester.run(df, signals)
        
        # 4. Metrics & Logging
        metrics = PerformanceMetrics.calculate_all(result.equity_curve)
        logger = ExperimentLogger()
        config = {
            "symbol": symbol_dropdown.value,
            "mode": mode_toggle.value,
            "threshold": threshold_slider.value,
            "capital": capital_input.value,
            "data_points": len(df)
        }
        log_path = logger.log(config, metrics)
        
        print(f"✅ Backtest Complete!")
        print(f"📊 Total Return: {metrics['total_return']:.2%}")
        print(f"📓 Log saved to: {log_path}")
        
        # 5. Visualization
        viz = Visualizer()
        
        # Charts
        display(viz.plot_equity_curve(result))
        display(viz.plot_signals(df, signals))
        
        # Audit Tables
        viz.render_scorecard(metrics)
        viz.render_trade_log(result.trades)

run_button.on_click(on_run_clicked)